# Week 07 Learning Notebook

This notebook mirrors the daily lesson plan and includes runnable demo code cells.

In [ ]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 07 Day 01: Unsupervised problem framing

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Use unsupervised methods to detect latent structure and regime behavior in financial data.

## Continuity and Handoff
- Previous checkpoint: Week 06 Day 07: Portfolio Project
- Previous lesson file: content/week-06/day-07.md
- Today's deliverable: Prepare normalized feature space for unsupervised analysis.
- Next handoff target: Week 07 Day 02: K-means and hierarchical clustering
- Next lesson file: content/week-07/day-02.md

## Theory Concepts

### Concept 1: Label-free learning objectives
Label-free learning objectives should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Distance metrics and scaling
Distance metrics and scaling should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Interpretability constraints
Interpretability constraints should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Forward Target
$$
y_t=\mathbb{1}[r_{t+1}>0]
$$
Label must stay forward-looking.

### Formula 2: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
Convert score to probability.

### Formula 3: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
Baseline regression loss.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\hat{y}$: model prediction
- $\lambda$: regularization parameter
- $TP,FP,FN$: confusion counts

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Frame a regime-detection task without labeled outcomes.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Unsupervised problem framing'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = -0.400.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(--0.400)) = 0.401312.
Final answer: Predicted probability = 40.13%.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Prepare normalized feature space for unsupervised analysis.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
How do preprocessing choices distort clustering outcomes?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# ML demo: directional classifier from real SPY features
np.random.seed(701)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


# Week 07 Day 02: K-means and hierarchical clustering

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Use unsupervised methods to detect latent structure and regime behavior in financial data.

## Continuity and Handoff
- Previous checkpoint: Week 07 Day 01: Unsupervised problem framing
- Previous lesson file: content/week-07/day-01.md
- Today's deliverable: Run k-means for multiple k values and evaluate silhouette trend.
- Next handoff target: Week 07 Day 03: PCA for regime and factor hints
- Next lesson file: content/week-07/day-03.md

## Theory Concepts

### Concept 1: Centroid-based clustering
Centroid-based clustering should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Cluster validity intuition
Cluster validity intuition should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Hierarchical dendrogram interpretation
Hierarchical dendrogram interpretation should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
Convert score to probability.

### Formula 2: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
Baseline regression loss.

### Formula 3: Cross-Entropy
$$
\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
$$
Classification objective.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\hat{y}$: model prediction
- $\lambda$: regularization parameter
- $TP,FP,FN$: confusion counts

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Cluster synthetic market states and compare cluster compactness.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'K-means and hierarchical clustering'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = -0.100.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(--0.100)) = 0.475021.
Final answer: Predicted probability = 47.50%.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Run k-means for multiple k values and evaluate silhouette trend.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
What indicates cluster instability across rolling windows?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# ML demo: directional classifier from real SPY features
np.random.seed(702)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


# Week 07 Day 03: PCA for regime and factor hints

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Use unsupervised methods to detect latent structure and regime behavior in financial data.

## Continuity and Handoff
- Previous checkpoint: Week 07 Day 02: K-means and hierarchical clustering
- Previous lesson file: content/week-07/day-02.md
- Today's deliverable: Project features into principal-component space and label clusters.
- Next handoff target: Week 07 Day 04: Anomaly detection basics
- Next lesson file: content/week-07/day-04.md

## Theory Concepts

### Concept 1: Dimensionality reduction
Dimensionality reduction should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Variance concentration
Variance concentration should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Noise filtering tradeoffs
Noise filtering tradeoffs should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: MSE
$$
\mathcal{L}_{MSE}=\frac{1}{n}\sum_i(y_i-\hat{y}_i)^2
$$
Baseline regression loss.

### Formula 2: Cross-Entropy
$$
\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
$$
Classification objective.

### Formula 3: Ridge Penalty
$$
\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2
$$
Regularized stability control.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\hat{y}$: model prediction
- $\lambda$: regularization parameter
- $TP,FP,FN$: confusion counts

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Use PCA components as regime features and visualize transitions.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'PCA for regime and factor hints'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 0.200.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-0.200)) = 0.549834.
Final answer: Predicted probability = 54.98%.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Project features into principal-component space and label clusters.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
When can PCA hide tail-relevant behavior?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# ML demo: directional classifier from real SPY features
np.random.seed(703)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


# Week 07 Day 04: Anomaly detection basics

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Use unsupervised methods to detect latent structure and regime behavior in financial data.

## Continuity and Handoff
- Previous checkpoint: Week 07 Day 03: PCA for regime and factor hints
- Previous lesson file: content/week-07/day-03.md
- Today's deliverable: Build a simple anomaly score and threshold report.
- Next handoff target: Week 07 Day 05: Regime interpretation and actionability
- Next lesson file: content/week-07/day-05.md

## Theory Concepts

### Concept 1: Outlier scoring
Outlier scoring should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Isolation-based intuition
Isolation-based intuition should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: False-positive management
False-positive management should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Cross-Entropy
$$
\mathcal{L}_{CE}=-\frac{1}{n}\sum_i[y_i\log p_i+(1-y_i)\log(1-p_i)]
$$
Classification objective.

### Formula 2: Ridge Penalty
$$
\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2
$$
Regularized stability control.

### Formula 3: Forward Target
$$
y_t=\mathbb{1}[r_{t+1}>0]
$$
Label must stay forward-looking.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\hat{y}$: model prediction
- $\lambda$: regularization parameter
- $TP,FP,FN$: confusion counts

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Detect unusual return/volume combinations in synthetic stress data.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Anomaly detection basics'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 0.500.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-0.500)) = 0.622459.
Final answer: Predicted probability = 62.25%.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build a simple anomaly score and threshold report.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
How should anomaly alerts feed into risk workflows?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# ML demo: directional classifier from real SPY features
np.random.seed(704)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


# Week 07 Day 05: Regime interpretation and actionability

## Study Duration
- Planned effort: 4 hours

## 4-Hour Lesson Flow
- 60 minutes: concept breakdown and formula derivation
- 75 minutes: real-market case study with data alignment checks
- 60 minutes: step-by-step quantitative problem solving
- 45 minutes: coding walkthrough and output verification

## Why It Matters in Quant
Use unsupervised methods to detect latent structure and regime behavior in financial data.

## Continuity and Handoff
- Previous checkpoint: Week 07 Day 04: Anomaly detection basics
- Previous lesson file: content/week-07/day-04.md
- Today's deliverable: Create a regime summary table with strategy implications.
- Next handoff target: Week 07 Day 06: Revision Sprint
- Next lesson file: content/week-07/day-06.md

## Theory Concepts

### Concept 1: Linking clusters to strategy behavior
Linking clusters to strategy behavior should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 2: Regime-aware feature gating
Regime-aware feature gating should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

### Concept 3: Operational limits of unsupervised signals
Operational limits of unsupervised signals should be treated as a measurable component of 'Unsupervised learning and market regimes'. For this week, emphasize causal feature design, leakage control, and robust model validation. State the formula, verify units, test edge cases, and explain exactly how market regime shifts could break the assumption.

## Mathematical Foundations (LaTeX)
### Formula 1: Ridge Penalty
$$
\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2
$$
Regularized stability control.

### Formula 2: Forward Target
$$
y_t=\mathbb{1}[r_{t+1}>0]
$$
Label must stay forward-looking.

### Formula 3: Logistic Probability
$$
p=\frac{1}{1+e^{-z}}
$$
Convert score to probability.

## Symbol Definitions
- $P_t$: price at time $t$
- $r_t$: simple return
- $\mu$: expected return
- $\sigma$: volatility
- $\hat{y}$: model prediction
- $\lambda$: regularization parameter
- $TP,FP,FN$: confusion counts

## Real Trading Example
- Instruments: SPY, XLK, XLF, XLE
- Macro overlay (FRED): DFF, VIXCLS
- Suggested window: 2018-01-01 to 2026-03-31
- Day objective: Map cluster labels to momentum and mean-reversion performance differences.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull the listed FRED series and join strictly by release-aware timestamps.
3. Compute today's formulas and compare behavior in stress sub-periods.
4. Translate quantitative results into one explicit trading decision and one risk guardrail.
5. Validate that the decision is consistent with topic 'Regime interpretation and actionability'.

## Step-by-Step Solved Problems
### Solved Problem 1: Convert logit to probability
Given:
- Model score z = 0.800.
Solution:
1. $p=\frac{1}{1+e^{-z}}$.
2. p = 1/(1+exp(-0.800)) = 0.689974.
Final answer: Predicted probability = 69.00%.

### Solved Problem 2: Compute precision, recall, F1
Given:
- TP=50, FP=12, FN=18.
Solution:
1. $Precision=\frac{TP}{TP+FP}$.
2. Precision=0.8065.
3. $Recall=\frac{TP}{TP+FN}$.
4. Recall=0.7353.
5. $F1=\frac{2PR}{P+R}$.
6. F1=0.7692.
Final answer: Precision=80.65%, Recall=73.53%, F1=76.92%.

### Solved Problem 3: Compute ridge objective
Given:
- MSE = 0.0410, ||beta||_2^2 = 1.90, lambda = 0.06.
Solution:
1. $\mathcal{L}=\mathcal{L}_{MSE}+\lambda\|\beta\|_2^2$.
2. L = 0.0410 + 0.06*1.90 = 0.1550.
Final answer: Ridge objective = 0.1550.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Create a regime summary table with strategy implications.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
X_train, X_valid, y_train, y_valid = temporal_split(features, target)
model = fit_logistic_regression(X_train, y_train, l2=0.1)
proba = model.predict_proba(X_valid)[:, 1]
report = classification_report_from_thresholds(y_valid, proba)
```

## Practice Problems
1. Re-derive all formulas manually and explain each variable.
2. Re-run the real trading example using one alternate ticker.
3. Stress-test one assumption and write a risk-control rule.
4. Extend the code walkthrough with one new validation test.

## Reflection Question
Which regime label is least actionable and why?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Reflection logged in progress tracker


In [ ]:
# ML demo: directional classifier from real SPY features
np.random.seed(705)
spy = load_market_prices(['SPY'], start='2014-01-01')['SPY']
ret = spy.pct_change()

feat = pd.DataFrame(index=spy.index)
feat['lag_1'] = ret.shift(1)
feat['lag_5'] = ret.rolling(5).mean().shift(1)
feat['vol_20'] = ret.rolling(20).std().shift(1)
feat['macro_rate'] = load_fred_series('DFF', start='2014-01-01').reindex(feat.index).ffill()
target = (ret.shift(-1) > 0).astype(int)
dataset = feat.join(target.rename('target')).dropna()

split = int(0.8 * len(dataset))
train = dataset.iloc[:split]
test = dataset.iloc[split:]

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, roc_auc_score

    model = LogisticRegression(max_iter=500)
    model.fit(train.drop(columns=['target']), train['target'])
    proba = model.predict_proba(test.drop(columns=['target']))[:, 1]
    pred = (proba >= 0.5).astype(int)
    out = {
        'accuracy': float(accuracy_score(test['target'], pred)),
        'auc': float(roc_auc_score(test['target'], proba)),
    }
except Exception:
    # fallback classifier if sklearn is unavailable
    pred = (test['lag_1'] > 0).astype(int)
    out = {
        'accuracy': float((pred.values == test['target'].values).mean()),
        'auc': float('nan'),
    }

out


# Week 07 Day 06: Revision Sprint

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 07 Day 05: Regime interpretation and actionability
- Previous lesson file: content/week-07/day-05.md
- Today's deliverable: Revision checklist with corrected errors and next-week retest priorities.
- Next handoff target: Week 07 Day 07: Portfolio Project
- Next lesson file: content/week-07/day-07.md

## Revision Plan
- 30 minutes: active recall of weekday concepts
- 40 minutes: rebuild one code workflow from memory
- 30 minutes: error log triage and retest plan
- 20 minutes: update progress tracker and notes

## Focus Areas
- Re-derive silhouette and cluster-quality interpretation
- Rebuild one PCA-regime chart from memory
- List unsupervised failure modes and mitigations

## Revision Output
- [ ] Updated error log entries
- [ ] Weak concepts re-tested
- [ ] Next-week risk list prepared


# Week 07 Day 07: Portfolio Project

## Study Duration
- Planned effort: 2 hours

## Continuity and Handoff
- Previous checkpoint: Week 07 Day 06: Revision Sprint
- Previous lesson file: content/week-07/day-06.md
- Today's deliverable: Regime clustering notebook
- Next handoff target: Week 08 Day 01: Pipeline architecture and feature store design
- Next lesson file: content/week-08/day-01.md

## Project Title
Regime clustering notebook

## Problem Statement
Detect and interpret market regimes using clustering and PCA features.

## Data Sources
- Returns
- Rolling volatility
- Volume anomalies

## Implementation Steps
1. Engineer regime-sensitive features
2. Run clustering methods
3. Validate cluster quality
4. Map regimes to strategy behavior
5. Write actionable insights

## Evaluation Metrics
- Silhouette score
- Cluster persistence
- Interpretability
- Strategy differentiation

## Deliverables
- Notebook or script output
- One-page summary memo
- Tracker update with completion and lessons learned
